# L4. Contextual Filtering for Memory

Similarity alone can't tell this morning's coffee from yesterday's. This lesson adds metadata filters: a time window, and a semantic query plus a payload filter, both inside the same on-device query.

## 1. Memories with metadata

In [1]:
import json
from datetime import datetime, timezone

memories = json.load(open("../data/memories.json"))
notes = [m for m in memories if m["source_type"] in ("text", "voice")]

def when(ts):
    return datetime.fromtimestamp(ts, timezone.utc).strftime("%a %H:%M")

for m in notes[:8]:
    text = m.get("note") or m["transcript"]
    price = f"${m['price']:>5.2f}" if "price" in m else "     -"
    print(f"{when(m['timestamp'])}  [{m['category']:>8}]  {price}  {text[:44]}")
print("...")
print(len(notes), "text and voice memories")

Tue 08:18  [    food]       -  Great little coffee place on 5th with outdoo
Tue 10:00  [    work]       -  Standup with Sarah moved to Thursday to revi
Tue 09:00  [ errands]       -  Pick up dry cleaning before Friday, ticket i
Tue 11:30  [    work]       -  Idea: batch the weekly report so it drafts i
Tue 19:00  [  social]       -  Mum's new address is 14 Elm Court, buzzer 3
Tue 15:18  [shopping]  $45.00  Liked the black and white running shoes at t
Tue 21:00  [  social]       -  Book club is reading the new sci-fi novel, w
Tue 12:00  [  health]       -  Dentist appointment confirmed for next Wedne
...
25 text and voice memories


## 2. Store, and index the fields we'll filter on

Passing a `filter` to the query is what makes filtering part of the search: recall becomes similarity *and* structure in one pass, not similarity followed by a second pass in your own code. Indexing the payload fields makes that filtering efficient: the engine uses the index to narrow candidates instead of scanning every point.

In [2]:
import shutil
from pathlib import Path
from qdrant_edge import EdgeShard, EdgeConfig, EdgeVectorParams, Distance

SHARD_DIR = "./mem_shard"
shutil.rmtree(SHARD_DIR, ignore_errors=True)
Path(SHARD_DIR).mkdir(parents=True, exist_ok=True)

config = EdgeConfig(
    vectors={
        "text": EdgeVectorParams(size=768, distance=Distance.Cosine),
    }
)
shard = EdgeShard.create(SHARD_DIR, config)

In [3]:
from qdrant_edge import Point, UpdateOperation
from helper import embed_text

docs = [m.get("note") or m["transcript"] for m in notes]
vectors = embed_text(docs)
shard.update(UpdateOperation.upsert_points([
    Point(id=m["id"], vector={"text": vectors[i]}, payload=m)
    for i, m in enumerate(notes)
]))
print("Stored", shard.info().points_count, "memories")

Stored 25 memories


In [4]:
from qdrant_edge import PayloadSchemaType

for field, kind in [
    ("category", PayloadSchemaType.Keyword),
    ("location", PayloadSchemaType.Keyword),
    ("timestamp", PayloadSchemaType.Float),
    ("price", PayloadSchemaType.Float),
]:
    shard.update(UpdateOperation.create_field_index(field, kind))
shard.optimize()
print("Indexed category, location, timestamp, price")

Indexed category, location, timestamp, price


## 3. The payoff: recall with a time window

In [5]:
from qdrant_edge import (
    Query, QueryRequest, Filter, FieldCondition, RangeFloat, MatchValue,
)
from helper import embed_query, before_after


def search(query_vector, query_filter=None, limit=4):
    hits = shard.query(
        QueryRequest(
            query=Query.Nearest(query_vector, using="text"),
            filter=query_filter,
            limit=limit,
            with_payload=True,
        )
    )
    return [h.payload.get("note") or h.payload["transcript"] for h in hits]


print("search(query_vector, filter) -> memory texts")

search(query_vector, filter) -> memory texts


In [6]:
BASE = 1_699_920_000
def at(hour): return BASE + int(hour * 3600)

query = "where did I get coffee"
query_vector = embed_query(query)

morning = Filter(
    must=[
        FieldCondition(key="timestamp", range=RangeFloat(gte=at(7), lte=at(12))),
    ]
)
before_after(query,
             "Similarity only", search(query_vector),
             "+ this morning (07:00-12:00)", search(query_vector, morning))

query: "where did I get coffee"

Similarity only:
  ✗ Just remembered, we are low on coffee at home, grab a bag on the way back
    Great little coffee place on 5th with outdoor seating and fast wifi
  ✗ Coffee run to the espresso bar near the office
  ✗ Found a quiet cafe with good wifi to work from near the park

+ this morning (07:00-12:00):
  ✓ Great little coffee place on 5th with outdoor seating and fast wifi
  ✓ New bakery on the corner does an amazing morning cronut
  ✓ Parked the bike near the station, second rack from the entrance
  ✓ Quick memo, the standup is moved to Thursday, tell the rest of the team


## 4. The payoff: recall with a semantic query plus a payload filter

In [7]:
query = "somewhere to eat"
query_vector = embed_query(query)

cheap_food = Filter(
    must=[
        FieldCondition(key="category", match=MatchValue(value="food")),
        FieldCondition(key="price", range=RangeFloat(lt=15)),
    ]
)
before_after(query,
             "Similarity only", search(query_vector),
             "+ food under $15", search(query_vector, cheap_food))

query: "somewhere to eat"

Similarity only:
  ✗ Great little coffee place on 5th with outdoor seating and fast wifi
  ✗ Found a quiet cafe with good wifi to work from near the park
    Note to self, the ramen downtown was incredible, fourteen dollars and worth it, sat right by the window
  ✗ Try the new ramen place downtown, everyone raves about the tonkotsu

+ food under $15:
  ✓ Note to self, the ramen downtown was incredible, fourteen dollars and worth it, sat right by the window
  ✓ Coffee run to the espresso bar near the office
  ✓ New bakery on the corner does an amazing morning cronut


## 5. Your turn: change the filter

In [8]:
# Change this and re-run. Try work, food, travel, shopping, home, health.
my_category = "work"

mine = Filter(
    must=[
        FieldCondition(key="category", match=MatchValue(value=my_category)),
    ]
)
query_vector = embed_query("what should I remember")
print(f"category = {my_category}:\n")
for text in search(query_vector, mine, limit=5):
    print(" ", text)

category = work:

  Quick memo, the standup is moved to Thursday, tell the rest of the team
  Meeting notes: ship the edge demo before the conference
  Standup with Sarah moved to Thursday to review the Q3 roadmap
  Idea: batch the weekly report so it drafts itself every Monday
  Found a quiet cafe with good wifi to work from near the park


## 6. The filter fields, for reference

| Field | Type | Example condition |
|---|---|---|
| `category` | keyword | `FieldCondition(key="category", match=MatchValue(value="food"))` |
| `price` | float | `FieldCondition(key="price", range=RangeFloat(lt=15))` |
| `timestamp` | float (epoch) | `FieldCondition(key="timestamp", range=RangeFloat(gte=start, lte=end))` |
| `location` | keyword | `FieldCondition(key="location", match=MatchValue(value="Office"))` |

> Combine conditions with `Filter(must=[...])` for AND, or `Filter(should=[...])` for OR.

In [9]:
from helper import cleanup
cleanup(shard, "./mem_shard")
print("Cleaned up")

Cleaned up
